In [1]:
import pandas as pd 
import numpy as np
import random
import os 
import argparse
import json
import torch
import pickle
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from torch.utils.data import TensorDataset
from attrdict import AttrDict
from transformers import BertConfig, BertTokenizer, BertModel
from transformers import DistilBertModel, DistilBertTokenizer, DistilBertConfig, DistilBertForSequenceClassification
from transformers import RobertaModel, RobertaTokenizer, RobertaConfig, RobertaForSequenceClassification
from transformers import AdamW, get_linear_schedule_with_warmup

In [3]:
default_path = os.getcwd()
base_model = os.path.join('C:/Users/lamda/Desktop/LAMDA_git/EmoDep/base-model')
config_path = 'C:/Users/lamda/Desktop/LAMDA_git/EmoDep/config'
model_path = "C:/Users/lamda/Desktop/LAMDA_git/EmoDep/model/data_aug/seed"
config_file = "bert-base.json"
save_path = os.path.join(default_path, './')

In [209]:
criteria = 9

In [210]:
test_data = pd.read_csv(f'C:/Users/lamda/Desktop/LAMDA_git/EmoDep/data/bws/tagged/train-test/bws_a{criteria}_score_test.csv')
print(len(test_data))
test_data.head(1)

320


,text,translated,label
0,I have gone from literally being suicidal to l...,나는 말 그대로 자살에서 나 자신을 사랑하고 내 삶을 진정으로 사랑하는 것으로 바뀌었다,1


#### Define Class 

In [211]:
class DSMDataset(Dataset):
    def __init__(self, data_file):
        self.data = data_file
    
    def __len__(self):
        return len(self.data.label)
    
    def reset_index(self):
        self.data.reset_index(inplace=True, drop=True)
    
    def __getitem__(self, idx):
        '''
        return text, label
        '''
        self.reset_index()
        text = self.data.text[idx]
        label = self.data.label[idx]
        return text, label

In [212]:
class DSMProcessor():
    def __init__(self, config, training_config, tokenizer, truncation=True):
        self.tokenizer = tokenizer 
        self.max_len =  config.max_position_embeddings
        self.pad = training_config.pad
        self.batch_size = training_config.train_batch_size
        self.truncation = truncation
    
    def convert_data(self, data_file):
        context2 = None    # single sentence classification
        batch_encoding = self.tokenizer.batch_encode_plus(
            [(data_file[idx][0], context2) for idx in range(len(data_file))],   # text, 
            max_length = self.max_len,
            padding = self.pad,
            truncation = self.truncation
        )
        
        features = []
        for i in range(len(data_file)):
            inputs = {k: batch_encoding[k][i] for k in batch_encoding}
            try:
                inputs['label'] = data_file[i][1] 
            except:
                inputs['label'] = 0 
            # print(inputs)
            features.append(inputs)
        
        all_input_ids = torch.tensor([f['input_ids'] for f in features], dtype=torch.long)
        all_attention_mask = torch.tensor([f['attention_mask'] for f in features], dtype=torch.long)
        # all_token_type_ids = torch.tensor([f['token_type_ids'] for f in features], dtype=torch.long)
        all_labels = torch.tensor([f['label'] for f in features], dtype=torch.long)

        dataset = TensorDataset(all_input_ids, all_attention_mask, all_labels)
        return dataset
    
    def shuffle_data(self, dataset, data_type):
        if data_type == 'train':
            return RandomSampler(dataset)
        elif data_type == 'eval' or data_type == 'test':
            return SequentialSampler(dataset)
        
    def load_data(self, dataset, sampler):
        return DataLoader(dataset, sampler=sampler, batch_size=self.batch_size)

In [213]:
class BertRegressor(nn.Module):
    def __init__(self, config, model):
        super(BertRegressor, self).__init__()
        self.model = model
        self.linear = nn.Linear(config.hidden_size, 128)
        self.relu = nn.ReLU()
        self.out = nn.Linear(128, 1)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.last_hidden_state[:, 0, :]
        # print(f'logits: {len(logits)}, {len(logits[0])}')
        x = self.linear(logits)
        x = self.relu(x)
        score = self.out(x)
        # print(f'score: {score}')
        return score 

In [214]:
class BertRegTester():
    def __init__(self, training_config, model):
        self.training_config = training_config
        self.model = model

    def get_label(self, test_dataloader, test_type):
        '''
        test_type: 0  -> Test dataset 
        test_type: 1  -> Test sentence
        '''
        preds = []
        labels = []

        for batch in test_dataloader:
            self.model.eval()   # self 안 붙이면 이상한 Output (BaseModelOutputWithPoolingAndCrossAttentions) 출력 
            batch = tuple(t.to(self.training_config.device) for t in batch)   # args.device: cuda 
            with torch.no_grad():
                inputs = {
                    "input_ids": batch[0],
                    "attention_mask": batch[1],
                    # "token_type_ids": batch[2],
                }
                outputs = self.model(**inputs)
                if test_type == 0:
                    # print(f'len preds: {len(preds)}')
                    try:
                        preds.extend(outputs.squeeze().detach().cpu().numpy())
                    except:
                        preds.extend(outputs[0].detach().cpu().numpy())
                elif test_type == 1:
                    preds.extend(outputs[0].detach().cpu().numpy())            
            label = batch[2].detach().cpu().numpy()
            labels.extend(label)
        return preds, labels 

In [215]:
def RMSELoss(yhat,y):
    return torch.sqrt(torch.mean((yhat-y)**2))

### Load Trained Model (BERT)

In [216]:
with open(os.path.join(config_path, 'training_config.json')) as f:
    training_config = AttrDict(json.load(f))

In [217]:
training_config.device = torch.device("cuda") if torch.cuda.is_available() else "cpu"
training_config.config_path = config_path
training_config.model_path = model_path 
training_config.pad = 'max_length'
training_config.num_epochs = 10

In [218]:
model_name = 'roberta-base'   # RobertaModel

In [219]:
tokenizer = RobertaTokenizer.from_pretrained(model_name, model_max_length=128)
config = RobertaConfig.from_pretrained(model_name)

In [220]:
model = RobertaModel.from_pretrained(model_name, config=config)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [221]:
reg_model = BertRegressor(config, model).to(training_config.device)

In [222]:
p_model = f'roberta_untagged_a{criteria}_s{seed}.pt'

In [223]:
reg_model = BertRegressor(config, model).to(training_config.device)

In [224]:
config.max_position_embeddings = 128
config.max_position_embeddings

128

In [225]:
reg_model.load_state_dict(torch.load(os.path.join(model_path, p_model)))
reg_model.to(training_config.device)

BertRegressor(
  (model): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0): RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,

### Data Transform 

In [226]:
dsm_processor = DSMProcessor(config, training_config, tokenizer)

In [227]:
test_file = DSMDataset(test_data)
test_dataset = dsm_processor.convert_data(test_file)
test_sampler = dsm_processor.shuffle_data(test_dataset, 'test')
test_dataloader = dsm_processor.load_data(test_dataset, test_sampler)

### Data Test 

In [228]:
bert_tester = BertRegTester(training_config, reg_model)

In [229]:
bws_pred, bws_true = bert_tester.get_label(test_dataloader, 0)

In [230]:
len(test_data)

320

In [231]:
criterion = RMSELoss
criterion(torch.Tensor(bws_pred), torch.Tensor(bws_true))

tensor(1.6877)

In [232]:
bws_pred, bws_true = bert_tester.get_label(test_dataloader, 0)

In [233]:
bws_pred = pd.DataFrame(bws_pred, columns=['pred'])
bws_pred.tail(3)

,pred
317,1.061763
318,4.045890
319,1.099827
